# Create Damon Runyon Awards (FELLOWSHIP PATTERN, Drupal field-scrape)

Damon Runyon Cancer Research Foundation is a US biomedical research
funder specializing in cancer (~$22M/yr, established 1946). Their
fellow database is published on `damonrunyon.org` (Drupal CMS) — 859
active + historical fellow profile pages exposed via the public
sitemap at `/sitemap/sitemap.xml`. Each profile carries consistent
`class="field field-name-field-X"` wrappers around institution,
mentor, award program, award type, cancer specialty, and research
area.

Fills another biomedical gap in the registry — Damon Runyon is the
largest cancer-research-specific funder not yet covered. Complements
HHMI (#44, biomedical generally) and NIH (#3, biomedical broadly).

**Awarding body:** Damon Runyon Cancer Research Foundation — F4320306271 (US)

**Schema choices:**
- One row per scientist (each fellow can hold only one Damon Runyon
  award at a time; `slug` is the canonical unique key).
- `funder_scheme` = `award_program` from the Drupal field (e.g.
  `'Damon Runyon Fellowship Award'`, `'Innovation Award'`,
  `'Clinical Investigator Award'`).
- `funding_type` derives from `award_type` via SQL CASE — fellowship,
  research, or other.
- `amount` and `currency` ship as NULL by design — Damon Runyon does
  not publicly disclose per-fellow amounts (program-level totals are
  public, e.g. $300K over 3 years for postdocs, but per-fellow
  allocations are not). Same waiver as HHMI (#44) and CIFAR.
- `lead_investigator` is the scientist; `affiliation.name` = the
  institution from the field.
- `cancer_type` and `research_area` carry through as metadata.
- `declined` boolean populated (always False; no declined-fellow data
  on the site).

**Source:** https://www.damonrunyon.org/sitemap/sitemap.xml + per-profile pages
**S3 location:** `s3a://openalex-ingest/awards/damon_runyon/damon_runyon_fellows.parquet`
**Method:** sitemap-driven Drupal field-extraction — different from
the REST-API patterns (Holberg, CIFAR, Nuffield) and the REST+landing
hybrid (Nuffield) and bulk CSV (NEH). Modeled here as a precedent for
any Drupal-rendered funder site with field-labeled markup.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.damon_runyon_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/damon_runyon/damon_runyon_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.damon_runyon_raw;

## Step 1.5: Inspect raw + §6.7 amount waiver

Damon Runyon does not publish per-fellow amounts (only program-level
totals). The parquet has no `amount` column by design — `amount` /
`currency` ship as NULL, and the §6.7 amount-coverage fail-fast is
**WAIVED** here with the same justification as HHMI (priority 44) and
CIFAR (priority 79).

Source columns: `funder_award_id`, `slug`, `scientist_full_name`,
`given_name`, `family_name`, `institution`, `sponsor_mentor`,
`award_program`, `award_type`, `cancer_type`, `research_area`,
`landing_page_url`, `declined`. All STRING per runbook §1.2.5.

In [ ]:
%sql
DESCRIBE openalex.awards.damon_runyon_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.damon_runyon_raw LIMIT 5;

In [ ]:
%sql
-- Coverage by award_program
SELECT award_program, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(sponsor_mentor) AS has_mentor
FROM openalex.awards.damon_runyon_raw
GROUP BY award_program
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast — verify Damon Runyon funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306271;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.damon_runyon_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306271  -- Damon Runyon Cancer Research Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Damon Runyon ', COALESCE(n.award_program, 'Award'), ' \u2014 ', n.scientist_full_name) AS display_name,
    CASE
        WHEN n.cancer_type IS NOT NULL AND n.research_area IS NOT NULL
            THEN CONCAT(n.award_type, ' working on ', n.cancer_type, ' (', n.research_area, ').',
                        CASE WHEN n.sponsor_mentor IS NOT NULL THEN CONCAT(' Mentor: ', n.sponsor_mentor, '.') ELSE '' END)
        WHEN n.cancer_type IS NOT NULL
            THEN CONCAT(n.award_type, ' working on ', n.cancer_type, '.',
                        CASE WHEN n.sponsor_mentor IS NOT NULL THEN CONCAT(' Mentor: ', n.sponsor_mentor, '.') ELSE '' END)
        WHEN n.award_type IS NOT NULL
            THEN n.award_type
        ELSE NULL
    END AS description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- funding_type via CASE on award_type
    CASE
        WHEN LOWER(n.award_type) RLIKE 'fellow' THEN 'fellowship'
        WHEN LOWER(n.award_type) RLIKE 'investigator|research|innovation|scholar' THEN 'research'
        WHEN LOWER(n.award_type) RLIKE 'clinical' THEN 'research'
        ELSE 'other'
    END AS funding_type,
    n.award_program AS funder_scheme,
    'damon_runyon_drupal' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CAST(NULL AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.scientist_full_name IS NULL OR n.scientist_full_name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                CAST('US' AS STRING) AS country,  -- Damon Runyon funds US-based researchers
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.damon_runyon_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.scientist_full_name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 82

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'damon_runyon_drupal' AND priority = 82;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    82 AS priority  -- Damon Runyon priority
FROM openalex.awards.damon_runyon_awards;

## Step 6: Verification

§6.7 amount-coverage check **WAIVED** for Damon Runyon (same
justification as HHMI #44 and CIFAR #79). Expect: ~850 rows,
pct_amount=0, currencies=[], 100% on name/program/funder.

In [ ]:
%sql
SELECT COUNT(*) AS total_damon_runyon_rows FROM openalex.awards.damon_runyon_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.damon_runyon_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_amount expected 0 (waiver).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.damon_runyon_awards;

In [ ]:
%sql
-- §6.7 — confirm waiver applied correctly. Expect all NULL.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.damon_runyon_awards;

In [ ]:
%sql
-- Sample
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS program, funding_type,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS institution
FROM openalex.awards.damon_runyon_awards
ORDER BY family
LIMIT 12;

In [ ]:
%sql
-- Program distribution
SELECT funder_scheme AS program, COUNT(*) AS rows
FROM openalex.awards.damon_runyon_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- funding_type split
SELECT funding_type, COUNT(*) AS rows
FROM openalex.awards.damon_runyon_awards
GROUP BY funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.damon_runyon_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;